# VLM-FO1 with a Qwen3.5 decoder — Colab

Runs the full pipeline (FO1 vision encoders + HFRE → bridge → Qwen3.5-0.8B) on a
free Colab GPU.

**Before you start:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.

The first run downloads about 10.5 GB of weights (VLM-FO1 8.9 GB, Qwen3.5 1.6 GB).

The feature bridge and region tokens are **untrained**, since no fine-tuning was in scope.
The pipeline runs end to end, but its answers are not meaningful detections.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code and install dependencies

In [ ]:
%cd /content
!test -d VLM || git clone -q -b qwen35-decoder-swap https://github.com/Milan4321123/VLM.git
%cd /content/VLM
!git log --oneline -1
# Pinned versions (torch 2.6, transformers 5.2, bitsandbytes). Takes a few minutes.
!pip install -q -r requirements-gpu.txt -r requirements-dev.txt

## 2. Offline test suite and preflight (no weights needed)

In [ ]:
!python -m pytest -q
!python cli.py --check

## 3. Full pipeline on the bundled fruit image

Seven supplied region proposals, seed 0, greedy decoding. The first run includes the download.

In [ ]:
!python inference.py --device cuda --max-new-tokens 64 \
    --output outputs/colab.jpg --json outputs/colab.json

In [ ]:
import json
from IPython.display import Image, display
result = json.load(open("outputs/colab.json"))
print(json.dumps({k: v for k, v in result.items() if k != "proposals"}, indent=2)[:3000])
display(Image("outputs/colab.jpg"))

## 4. Your own prompt

Change the task, target or boxes. Boxes are `[x1, y1, x2, y2]` in pixels.

In [ ]:
!python cli.py --image demo/demo_image.jpg --task od --target "orange" \
    --boxes '[[161,11,292,127],[268,61,428,226],[12,100,140,227]]' \
    --device cuda --max-new-tokens 64 --output outputs/custom.jpg

## 5. Optional: 4-bit decoder (uses less GPU memory)

In [ ]:
!python inference.py --device cuda --load-4bit --max-new-tokens 64 \
    --output outputs/colab-4bit.jpg --json outputs/colab-4bit.json

## 6. Web demo

Prints a public `https://….gradio.live` link. Open it in a new tab.
Stop the cell to shut the demo down.

In [ ]:
!GRADIO_SHARE=1 python demo/gradio_demo.py